# RoofD training on TIDE

Run this notebook from a TIDE JupyterHub **pytorch-notebook** session (default: 8 CPU / 32 GB RAM / **L40**).

**Order:** setup → (optional smoke) → download → prepare → logistic → ViT → U-Net → evaluate.

This notebook drives the current pipeline: group-aware train/val split (`group_by: tile`), ViT fine-tuning of the trailing six blocks with layer-wise LRs and validation-tuned per-class thresholds, U-Net with Focal + Dice and a cosine schedule, and ViT + U-Net ensemble evaluation calibrated on validation fusion.

Full ViT + U-Net training can take a long time. Prefer a **batch job** for overnight runs if the browser session may disconnect. Train on CUDA; a CPU-only host needs tens of hours.

> Research demo only — not for certified structural inspection.

## 1. Repo root and GPU check

**Re-run this section after any kernel restart.** It locates the repo root, sets `PYTHONPATH`, and defines `run_module()` so training cells do not depend on fragile `!python -m` shell cwd.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root() -> Path:
    """Walk up from cwd / this notebook until configs + src/train_vit.py exist."""
    here = Path.cwd().resolve()
    candidates = [here, here.parent, *here.parents]
    # Also try common TIDE clone layouts
    candidates += [
        Path("/home/jovyan/UAVRoofDetectionVIT"),
        Path.home() / "UAVRoofDetectionVIT",
    ]
    for path in candidates:
        if (path / "configs" / "default.yaml").exists() and (path / "src" / "train_vit.py").exists():
            return path
    raise FileNotFoundError(
        "Could not find repo root (need configs/default.yaml and src/train_vit.py). "
        f"cwd={here}"
    )

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
os.environ["PYTHONPATH"] = str(REPO_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

def run_module(module: str, *args: str) -> None:
    """Run `python -m <module>` with an explicit cwd + PYTHONPATH (shell magics are unreliable)."""
    cmd = [sys.executable, "-m", module, *args]
    print("$", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(REPO_ROOT), env=os.environ.copy())

print("Repo root:", REPO_ROOT)
print("python:", sys.executable)
print("cwd:", Path.cwd())

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / (1024**3):.1f} GB")
else:
    print("WARNING: no CUDA — use CPU only for smoke tests, not full training.")

## 2. Install project dependencies

On TIDE, keep the image’s CUDA PyTorch; `requirements.txt` only adds the rest (`timm`, SMP, OpenCV, …).

In [ ]:
# Must use the same interpreter as training (`sys.executable`).
%pip install -q opencv-python-headless timm segmentation-models-pytorch scikit-learn scikit-image streamlit PyYAML matplotlib pandas

import cv2
import timm
import segmentation_models_pytorch as smp
print("cv2", cv2.__version__)
print("timm", timm.__version__)
print("smp", smp.__version__)

## 3. Optional smoke test (no RoofD download)

Skip this section once the pipeline is verified. Smoke mode uses 1 epoch, batch 2, no pretrained weights, and writes to `outputs/smoke`. It exercises the group-aware split loader, per-class threshold tuning, the U-Net Focal + Dice path, and validation-calibrated ensemble fusion without touching the real artifacts.

In [ ]:
RUN_SMOKE = False  # set True to verify the end-to-end path on synthetic data

if RUN_SMOKE:
    run_module("scripts.make_smoke_data")
    run_module("src.train_logistic", "--smoke", "--limit", "32", "--data-root", "/tmp/roofd_smoke")
    run_module("src.train_vit", "--smoke", "--limit", "32", "--data-root", "/tmp/roofd_smoke")
    run_module("src.train_unet", "--smoke", "--limit", "32", "--data-root", "/tmp/roofd_smoke")
    run_module("src.evaluate", "--smoke", "--limit", "32", "--data-root", "/tmp/roofd_smoke", "--model", "all")
else:
    print("Smoke test skipped (RUN_SMOKE=False).")

## 4. Download RoofD (~6.2 GB)

Run once per storage location. Reuses `data/raw/RoofD` if already present; the archive extracts to about 20 GB.

In [ ]:
from pathlib import Path

roofd = REPO_ROOT / "data/raw/RoofD"
if roofd.exists() and any(roofd.rglob("*.png")):
    print("RoofD already present at", roofd.resolve())
else:
    print("Downloading RoofD (this can take a while)...")
    run_module("scripts.download_roofd")

## 5. Prepare data (verify + cache stats + group-aware split)

Writes `outputs/splits.json`, `outputs/metrics/train_stats.json`, `outputs/metrics/test_stats.json`, `outputs/metrics/dataset_summary.json`, and sample figures.

The split is group-aware: `data.group_by: tile` keeps all four pre-augmented copies (`0000_`–`0003_`) of a tile on one side of train/val; `flight` holds out entire roofs. Filename-only splitting leaks augmented copies into validation and inflates validation F1. **Re-run this after pulling any data or split change**, before training.

Some RoofD masks contain off-palette RGB edge colors; the loader remaps them to the nearest class and prints a one-time warning per color.

In [ ]:
run_module("scripts.prepare_data", "--verify-images")

## 6. Train logistic baseline

Fast CPU-friendly baseline. Checkpoint: `outputs/checkpoints/logistic.joblib`.

In [ ]:
run_module("src.train_logistic")

## 7. Train Vision Transformer (primary)

Staged fine-tune: 5-epoch frozen-backbone head warm-up, then 15 epochs on the trailing **six** transformer blocks with layer-wise LRs (`1e-5` blocks 6–8, `2e-5` blocks 9–11, `1e-4` head), `drop_path_rate=0.2`, and `BCEWithLogitsLoss` with √-damped `pos_weight` capped at 10. Needs CUDA / L40.

After training the script reloads the best-validation checkpoint, sweeps per-class thresholds on validation (`0.10`–`0.90`, F1-maximizing), stores them under `thresholds` in the checkpoint, and reports test metrics with those thresholds. Uses the leak-free group-aware split, so validation macro-F1 is now an honest generalization estimate.

Checkpoint: `outputs/checkpoints/vit_best.pt`

In [ ]:
run_module("src.train_vit")

## 8. Train U-Net (partner pipeline)

ResNet34 ImageNet encoder, 256×256 input, 40 epochs, Focal loss (`gamma=2`, log-smoothed class weights) + Dice loss weighted equally, Adam at `1e-4` with `CosineAnnealingLR` (`eta_min=1e-6`). Checkpoint: `outputs/checkpoints/unet_best.pt`

In [ ]:
run_module("src.train_unet")

## 9. Evaluate on test set

`--model all` evaluates ViT, U-Net, logistic, and the ensemble. The ensemble pools each U-Net softmax mask into image-level presence (top 100/100/40/30/20 pixels) and fuses it with ViT probabilities using `alpha=[0.7, 0.7, 0.7, 0.3, 0.2]` from `configs/default.yaml`. Because fused probabilities have their own calibration, it sweeps per-class thresholds on **validation** fusion and writes `outputs/metrics/ensemble_thresholds.json`; the Streamlit demo reads that file.

Set `RUN_DEGRADATION = True` to also run the UAV resolution experiment (slower).

In [ ]:
RUN_DEGRADATION = False

run_module("src.evaluate", "--model", "all", "--split", "test")

if RUN_DEGRADATION:
    run_module("src.evaluate", "--model", "all", "--split", "test", "--degradation")
else:
    print("Degradation experiment skipped (RUN_DEGRADATION=False).")

## 10. Checkpoint / metric summary

In [ ]:
from pathlib import Path
import json

ckpts = Path("outputs/checkpoints")
metrics = Path("outputs/metrics")

print("Checkpoints:")
for p in sorted(ckpts.glob("*")):
    print(f"  {p.name:24s} {p.stat().st_size / 1e6:8.1f} MB")

for name in ("logistic_test.json", "vit_test.json", "unet_test.json", "ensemble_test.json"):
    path = metrics / name
    if not path.exists():
        print(f"Missing {path}")
        continue
    data = json.loads(path.read_text())
    if "macro_f1" in data:
        print(f"{name}: macro_f1={data['macro_f1']:.4f}")
        if "thresholds" in data:
            print("    thresholds:", data["thresholds"])
    elif "miou" in data:
        print(f"{name}: mIoU={data['miou']:.4f}")

fusion_path = metrics / "ensemble_thresholds.json"
if fusion_path.exists():
    fusion = json.loads(fusion_path.read_text())
    print("Fusion thresholds:", fusion["thresholds"], "| alpha:", fusion["alpha"])
    print(f"Validation fusion macro-F1: {fusion['val_macro_f1']:.4f}")
else:
    print("Missing", fusion_path, "(run evaluate --model all)")

## Optional: Streamlit demo

In a terminal on TIDE (after checkpoints exist):

```bash
cd /path/to/UAVRoofDetectionVIT
streamlit run src/app.py
```